## NBA Road Trip A* Optimizer

This notebook is fully self-contained: data generation, cost modeling, exact A* search, performance reporting, and visualization all live in `notebooks-optimization/road_trip_astar_optimizer.ipynb`.

### 1. Setup

In [ ]:
# Cell 1: Environment Setup & Imports

import heapq
import math
import time
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import folium
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import HTML, Markdown, display

# Configure Polars display settings for improved data visualization
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(25)
pl.Config.set_float_precision(2)

# Output initialization status to confirm all required libraries are loaded
print("=" * 80)
print("Environment ready: Polars, NumPy, NetworkX, Plotly, Folium, and heapq imported.")
print("=" * 80)

### 2. Dataset Preparation and Cost Matrix

In this phase, we set up the data infrastructure for the optimization algorithm:

* **The "Home" Node**: We define `Home` as our starting and ending point, located at the **Chase Center (San Francisco)**. This node acts as a logistical anchor: the route must start here, visit every other stadium exactly once, and return here.
* **Data Management**: We use **Polars** to build a comprehensive cost matrix. For every pair of cities, we calculate:
* **Haversine Distance**: The real geodesic distance on the Earth's surface.
* **Total Cost**: A function composed of `(Distance * 1.5) + Fixed Logistical Cost`.


* **Adjacency Matrix**: We transform this data into an $N \times N$ square matrix, where rows represent origins and columns represent destinations. Each cell contains the transition cost between two stadiums. This structure allows the A* algorithm to instantly query the cost of any move while searching for the optimal route.

In [ ]:
# Cell 2: Dataset Ingestion & Processing with Polars

# Constants for logistical calculations
EARTH_RADIUS_KM = 6371.0088
TRAVEL_COST_PER_KM = 1.5
HOME_CITY = "Home"

# Initialize Polars DataFrame with stadium geographical coordinates and fixed logistical costs
stadiums = pl.DataFrame(
    {
        "City_Name": [
            "Home", "Los Angeles", "Phoenix", "Denver", "Dallas", "Houston", 
            "New Orleans", "Memphis", "Atlanta", "Miami", "Charlotte", 
            "Washington DC", "Philadelphia", "New York", "Boston", 
            "Toronto", "Chicago", "Milwaukee",
        ],
        "Latitude": [
            37.7680, 34.0430, 33.4457, 39.7487, 32.7905, 29.7508, 
            29.9490, 35.1382, 33.7573, 25.7814, 35.2251, 38.8981, 
            39.9012, 40.7505, 42.3662, 43.6435, 41.8807, 43.0451,
        ],
        "Longitude": [
            -122.3877, -118.2673, -112.0712, -105.0077, -96.8103, -95.3621, 
            -90.0821, -90.0506, -84.3963, -80.1870, -80.8392, -77.0209, 
            -75.1720, -73.9934, -71.0621, -79.3791, -87.6742, -87.9172,
        ],
        "Fixed_Logistical_Cost": [
            0.0, 42500.0, 31500.0, 35500.0, 38250.0, 36750.0, 34250.0, 
            30500.0, 37750.0, 46250.0, 32750.0, 39750.0, 40250.0, 51500.0, 
            48750.0, 44500.0, 41750.0, 33250.0,
        ],
    }
)

# Prepare two projections of the stadium table to perform a cross join
# This allows for the calculation of all possible directed city-to-city paths
origin = stadiums.select(
    pl.col("City_Name").alias("Origin"),
    pl.col("Latitude").alias("Origin_Latitude"),
    pl.col("Longitude").alias("Origin_Longitude"),
)

destination = stadiums.select(
    pl.col("City_Name").alias("Destination"),
    pl.col("Latitude").alias("Destination_Latitude"),
    pl.col("Longitude").alias("Destination_Longitude"),
    pl.col("Fixed_Logistical_Cost").alias("Destination_Fixed_Logistical_Cost"),
)

# Calculate pairwise travel costs using the Haversine formula
pairwise_costs = (
    origin.join(destination, how="cross")
    .with_columns(
        [
            pl.col("Origin_Latitude").radians().alias("origin_lat_rad"),
            pl.col("Destination_Latitude").radians().alias("destination_lat_rad"),
            (pl.col("Destination_Latitude") - pl.col("Origin_Latitude")).radians().alias("delta_lat_rad"),
            (pl.col("Destination_Longitude") - pl.col("Origin_Longitude")).radians().alias("delta_lon_rad"),
        ]
    )
    .with_columns(
        (
            (pl.col("delta_lat_rad") / 2).sin().pow(2)
            + pl.col("origin_lat_rad").cos()
            * pl.col("destination_lat_rad").cos()
            * (pl.col("delta_lon_rad") / 2).sin().pow(2)
        ).alias("haversine_a")
    )
    .with_columns(
        (2 * EARTH_RADIUS_KM * pl.col("haversine_a").sqrt().arcsin()).alias("Distance_in_km")
    )
    .with_columns(
        pl.when(pl.col("Origin") == pl.col("Destination"))
        .then(0.0)
        .otherwise((pl.col("Distance_in_km") * TRAVEL_COST_PER_KM) + pl.col("Destination_Fixed_Logistical_Cost"))
        .alias("Total_Cost")
    )
    .select(
        [
            "Origin",
            "Destination",
            "Distance_in_km", # Preserving full precision for A* algorithm stability
            "Destination_Fixed_Logistical_Cost",
            "Total_Cost",     # Preserving full precision to avoid rounding errors
        ]
    )
)

# Pivot the data into a square cost matrix for graph adjacency representation
cost_matrix = (
    pairwise_costs.pivot(
        values="Total_Cost",
        index="Origin",
        on="Destination", 
        aggregate_function="first",
    )
    .select(["Origin", *stadiums.get_column("City_Name").to_list()])
    .sort("Origin")
)

# Log processing status
print("=" * 80)
print(f"Loaded {stadiums.height} NBA stadium/city nodes, including `{HOME_CITY}` as the franchise origin.")
print("Cost formula: Total_Cost = (Distance_in_km * 1.5) + Fixed_Logistical_Cost_of_Destination")
print("=" * 80)

# Display a formatted version of the matrix for user verification
display_matrix = cost_matrix.with_columns(pl.exclude("Origin").round(2))
print("Pairwise Total Cost Matrix:")
display(display_matrix)

### 3. A* State and Heuristic Definition

Each search state is the immutable tuple `(current_node, unvisited_cities_tuple)`. 
The path cost `g(n)` is the exact accumulated directed cost from `Home` to `current_node`.

For a state with remaining city set `R`, the heuristic is: `h(n) = MST(R) + min_distance(current_node, R) + min_distance(Home, R) + sum(destination_fixed_costs_for_R)`

The distance component is a lower bound because any feasible completion must connect the current city, all remaining cities, and Home in one connected structure; removing one edge from the eventual completion 
leaves a spanning structure whose cost is at least the MST plus the cheapest admissible endpoint connections. 

The fixed logistical costs for unvisited destinations are mandatory exactly once, independent of order. Therefore the heuristic never overestimates. Because travel distances are metric Haversine distances and fixed destination costs drop by exactly the amount paid when a city is visited, the lower bound is monotone across state transitions, so A* can safely use a best-cost dictionary for duplicate states.

In [ ]:
# Cell 3: The A* Search Algorithm Core

# Define type aliases for state management and route representation
State = Tuple[str, Tuple[str, ...]]
Route = List[str]

# Extract basic data structures from the previously defined stadiums DataFrame
city_names: List[str] = stadiums.get_column("City_Name").to_list()
fixed_logistical_cost: Dict[str, float] = {
    row["City_Name"]: float(row["Fixed_Logistical_Cost"])
    for row in stadiums.iter_rows(named=True)
}

# Create cost lookups for efficient retrieval during the search process
cost_lookup: Dict[Tuple[str, str], float] = {
    (row["Origin"], row["Destination"]): float(row["Total_Cost"])
    for row in pairwise_costs.iter_rows(named=True)
}

travel_cost_lookup: Dict[Tuple[str, str], float] = {
    (row["Origin"], row["Destination"]): float(row["Distance_in_km"]) * TRAVEL_COST_PER_KM
    for row in pairwise_costs.iter_rows(named=True)
}


def canonical_unvisited(cities: Iterable[str]) -> Tuple[str, ...]:
    """Return a stable, immutable representation of the set of unvisited cities.

    Parameters
    ----------
    cities:
        An iterable of city names to be converted into the remaining TSP set.

    Returns
    -------
    tuple[str, ...]
        A sorted tuple, suitable for hashing and consistent A* state identification.

    Complexity
    ----------
    O(k log k) time and O(k) space, where k is the number of cities.
    """
    return tuple(sorted(cities))


@lru_cache(maxsize=None)
def minimum_spanning_tree_travel_cost(unvisited_cities_tuple: Tuple[str, ...]) -> float:
    """Compute the Minimum Spanning Tree (MST) travel lower bound for unvisited cities.

    Parameters
    ----------
    unvisited_cities_tuple:
        An immutable, canonical tuple of remaining city names.

    Returns
    -------
    float
        The MST cost over remaining cities (distance-based only). Destination fixed
        logistical costs are handled separately by the A* heuristic.

    Complexity
    ----------
    Uses NetworkX's Prim's algorithm. O(r^2 log r) for r cities. Cached via LRU.
    """
    remaining = tuple(unvisited_cities_tuple)

    if len(remaining) <= 1:
        return 0.0

    graph = nx.Graph()
    graph.add_nodes_from(remaining)

    for origin_index, origin_city in enumerate(remaining):
        for destination_city in remaining[origin_index + 1 :]:
            graph.add_edge(
                origin_city,
                destination_city,
                weight=travel_cost_lookup[(origin_city, destination_city)],
            )

    mst = nx.minimum_spanning_tree(graph, algorithm="prim", weight="weight")
    return float(mst.size(weight="weight"))


@lru_cache(maxsize=None)
def astar_heuristic(current_node: str, unvisited_cities_tuple: Tuple[str, ...]) -> float:
    """Estimate the minimum remaining cost from the current state to a completed tour.

    Parameters
    ----------
    current_node:
        The current city location of the franchise.
    unvisited_cities_tuple:
        The set of cities yet to be visited.

    Returns
    -------
    float
        An admissible/consistent lower bound for the remaining travel cost.

    Complexity
    ----------
    O(r^2 log r) on cache miss; O(r) on cache hit.
    """
    remaining = tuple(unvisited_cities_tuple)

    if not remaining:
        return cost_lookup[(current_node, HOME_CITY)]

    mandatory_fixed_cost = sum(fixed_logistical_cost[city] for city in remaining)
    mandatory_home_return_fixed_cost = fixed_logistical_cost[HOME_CITY]
    mst_lower_bound = minimum_spanning_tree_travel_cost(remaining)
    nearest_current_connection = min(travel_cost_lookup[(current_node, city)] for city in remaining)
    nearest_home_connection = min(travel_cost_lookup[(HOME_CITY, city)] for city in remaining)

    return (
        mandatory_fixed_cost
        + mandatory_home_return_fixed_cost
        + mst_lower_bound
        + nearest_current_connection
        + nearest_home_connection
    )


def compute_route_total_cost(route: Sequence[str]) -> float:
    """Compute the exact directed cost of a full or partial route.

    Parameters
    ----------
    route:
        An ordered sequence of city names.

    Returns
    -------
    float
        The sum of all directed travel and logistical leg costs.
    """
    return float(sum(cost_lookup[(route[index], route[index + 1])] for index in range(len(route) - 1)))


def nearest_neighbor_upper_bound(start_city: str, all_cities: Sequence[str]) -> Tuple[Route, float]:
    """Construct a greedy feasible TSP tour for branch-and-bound pruning.

    Returns
    -------
    tuple[list[str], float]
        A feasible Home-to-Home tour and its exact cost (incumbent upper bound).
    """
    current_city = start_city
    remaining = set(city for city in all_cities if city != start_city)
    route = [start_city]

    while remaining:
        next_city = min(remaining, key=lambda city: (cost_lookup[(current_city, city)], city))
        route.append(next_city)
        remaining.remove(next_city)
        current_city = next_city

    route.append(start_city)
    return route, compute_route_total_cost(route)


def improve_route_with_2opt(route: Route) -> Tuple[Route, float]:
    """Refine a feasible tour using deterministic 2-opt swaps to lower the upper bound.

    Complexity
    ----------
    O(i * n^3) time, where n is the number of cities and i is the number of passes.
    """
    best_route = list(route)
    best_cost = compute_route_total_cost(best_route)
    improved = True

    while improved:
        improved = False
        for left in range(1, len(best_route) - 2):
            for right in range(left + 1, len(best_route) - 1):
                candidate_route = best_route[:left] + best_route[left : right + 1][::-1] + best_route[right + 1 :]
                candidate_cost = compute_route_total_cost(candidate_route)
                if candidate_cost + 1e-9 < best_cost:
                    best_route = candidate_route
                    best_cost = candidate_cost
                    improved = True
                    break
            if improved:
                break

    return best_route, best_cost


def solve_road_trip_astar(start_city: str, all_cities: Sequence[str]) -> Tuple[Route, float, Dict[str, float]]:
    """Solve the road-trip TSP exactly using the A* search algorithm.

    Returns
    -------
    tuple[list[str], float, dict[str, float]]
        Optimal route, total cost, and performance metrics.
    """
    initial_unvisited = canonical_unvisited(city for city in all_cities if city != start_city)
    initial_state: State = (start_city, initial_unvisited)

    greedy_route, greedy_cost = nearest_neighbor_upper_bound(start_city, all_cities)
    incumbent_route, incumbent_cost = improve_route_with_2opt(greedy_route)

    priority_queue: List[Tuple[float, float, int, State, Route]] = []
    tie_breaker = 0
    initial_g_cost = 0.0
    initial_f_cost = initial_g_cost + astar_heuristic(*initial_state)

    # Priority queue stores (f, g, tie_breaker, state, path)
    heapq.heappush(priority_queue, (initial_f_cost, initial_g_cost, tie_breaker, initial_state, [start_city]))

    # Dictionary to track the minimum g(n) for each state to prune inferior paths
    best_g_by_state: Dict[State, float] = {initial_state: initial_g_cost}

    expanded_states = 0
    enqueued_states = 1
    skipped_stale_states = 0
    pruned_transitions = 0
    max_queue_size = 1

    while priority_queue:
        current_f_cost, current_g_cost, _, current_state, current_path = heapq.heappop(priority_queue)
        current_node, unvisited_cities_tuple = current_state

        # Skip if a cheaper path to this state was already processed
        if current_g_cost > best_g_by_state.get(current_state, math.inf) + 1e-9:
            skipped_stale_states += 1
            continue

        # Prune if the current lower bound cannot improve upon the incumbent solution
        if current_f_cost >= incumbent_cost - 1e-9 and unvisited_cities_tuple:
            pruned_transitions += 1
            continue

        expanded_states += 1

        if not unvisited_cities_tuple:
            completed_route = current_path + [start_city]
            completed_cost = current_g_cost + cost_lookup[(current_node, start_city)]
            if completed_cost <= incumbent_cost + 1e-9:
                incumbent_route = completed_route
                incumbent_cost = completed_cost
            
            # Aggregate final diagnostics
            metrics = {
                "expanded_states": float(expanded_states),
                "enqueued_states": float(enqueued_states),
                "skipped_stale_states": float(skipped_stale_states),
                "pruned_transitions": float(pruned_transitions),
                "max_queue_size": float(max_queue_size),
                "mst_cache_hits": float(minimum_spanning_tree_travel_cost.cache_info().hits),
                "mst_cache_misses": float(minimum_spanning_tree_travel_cost.cache_info().misses),
                "heuristic_cache_hits": float(astar_heuristic.cache_info().hits),
                "heuristic_cache_misses": float(astar_heuristic.cache_info().misses),
                "incumbent_upper_bound": float(greedy_cost),
            }
            return incumbent_route, incumbent_cost, metrics

        # Expand successor states
        candidate_cities = sorted(
            unvisited_cities_tuple,
            key=lambda city: (cost_lookup[(current_node, city)], city),
        )

        for next_city in candidate_cities:
            next_unvisited = tuple(city for city in unvisited_cities_tuple if city != next_city)
            next_state: State = (next_city, next_unvisited)
            tentative_g_cost = current_g_cost + cost_lookup[(current_node, next_city)]

            if tentative_g_cost >= best_g_by_state.get(next_state, math.inf) - 1e-9:
                continue

            next_h_cost = astar_heuristic(next_city, next_unvisited)
            next_f_cost = tentative_g_cost + next_h_cost

            if next_f_cost >= incumbent_cost - 1e-9:
                pruned_transitions += 1
                continue

            best_g_by_state[next_state] = tentative_g_cost
            tie_breaker += 1

            heapq.heappush(
                priority_queue,
                (
                    next_f_cost,
                    tentative_g_cost,
                    tie_breaker,
                    next_state,
                    current_path + [next_city],
                ),
            )
            enqueued_states += 1

        max_queue_size = max(max_queue_size, len(priority_queue))

    # Return final metrics if the queue is exhausted
    return incumbent_route, incumbent_cost, {
        "expanded_states": float(expanded_states),
        "enqueued_states": float(enqueued_states),
        "pruned_transitions": float(pruned_transitions),
        "max_queue_size": float(max_queue_size),
        "mst_cache_hits": float(minimum_spanning_tree_travel_cost.cache_info().hits),
        "heuristic_cache_hits": float(astar_heuristic.cache_info().hits),
        "incumbent_upper_bound": float(greedy_cost),
    }

print("=" * 80)
print("A* core ready. State = (current_node, unvisited_cities_tuple); duplicate states tracked by min g(n).")
print("=" * 80)

### 4. Execution, Performance, and Analysis

Once the infrastructure is set, we execute the search and interpret the results through detailed diagnostics and visualization:

* **A* Engine Execution**: We run the `solve_road_trip_astar` algorithm. By clearing the caches before execution, we ensure a clean baseline to measure how effectively the heuristic guides the search and how often the MST (Minimum Spanning Tree) and heuristic functions are computed.
* **Performance Metrics**: We monitor key indicators to assess the algorithm's efficiency:
* **Expanded/Enqueued States**: Measures how much of the "state space" was actually explored versus how many nodes were discovered.
* **Pruned Transitions**: Tracks how many sub-optimal paths were discarded early, demonstrating the power of the Branch & Bound logic.
* **Cache Hits/Misses**: Validates the effectiveness of our memoization strategy, which is critical for maintaining high performance.


* **Dual-Axis Visualization**: We generate a professional Plotly chart to analyze the cost structure. The **bar trace** displays the "Incremental Cost" of each specific leg (travel + fixed fees), while the **line trace** plots the "Cumulative Cost," providing an intuitive view of how total expenditure grows as the road trip progresses.

In [ ]:
# Cell 4: Execution, Performance Metrics, and Incremental Cost Visualization

# Reset caches to ensure a clean execution baseline
minimum_spanning_tree_travel_cost.cache_clear()
astar_heuristic.cache_clear()

# Execute A* search and measure performance
start_time = time.perf_counter()
optimal_route, optimal_total_cost, search_metrics = solve_road_trip_astar(HOME_CITY, city_names)
execution_time_ms = (time.perf_counter() - start_time) * 1000

# Compile per-leg statistics for post-search analysis
route_leg_records = []
cumulative_cost = 0.0

for leg_number, (origin_city, destination_city) in enumerate(zip(optimal_route[:-1], optimal_route[1:]), start=1):
    incremental_cost = cost_lookup[(origin_city, destination_city)]
    cumulative_cost += incremental_cost
    route_leg_records.append(
        {
            "Leg": leg_number,
            "Origin": origin_city,
            "Destination": destination_city,
            "Incremental_Cost": round(incremental_cost, 2),
            "Cumulative_Cost": round(cumulative_cost, 2),
            "Distance_Travel_Cost": round(travel_cost_lookup[(origin_city, destination_city)], 2),
            "Destination_Fixed_Logistical_Cost": round(fixed_logistical_cost[destination_city], 2),
        }
    )

# Format the results into a Polars DataFrame and generate a path string
route_legs = pl.DataFrame(route_leg_records)
route_label = " -> ".join(optimal_route)

# Print performance summary to standard output
print("=" * 80)
print("OPTIMAL NBA ROAD TRIP TOUR")
print("=" * 80)
print(route_label)
print("-" * 80)
print(f"Total optimal cost: ${optimal_total_cost:,.2f}")
print(f"Execution time: {execution_time_ms:,.2f} ms")
print(f"Expanded states: {int(search_metrics['expanded_states']):,}")
print(f"Enqueued states: {int(search_metrics['enqueued_states']):,}")
print(f"Pruned transitions/states: {int(search_metrics['pruned_transitions']):,}")
print(f"Maximum priority queue size: {int(search_metrics['max_queue_size']):,}")
print(f"MST cache hits/misses: {int(search_metrics['mst_cache_hits']):,} / {int(search_metrics['mst_cache_misses']):,}")
print(f"Heuristic cache hits/misses: {int(search_metrics['heuristic_cache_hits']):,} / {int(search_metrics['heuristic_cache_misses']):,}")
print("=" * 80)

# Display tabular data for detailed route breakdown
display(HTML("<h3>Optimized Route Leg Cost Breakdown</h3>"))
display(HTML(route_legs._repr_html_()))

# Generate interactive visualizations using Plotly
leg_labels = [
    f"{row['Leg']:02d}. {row['Origin']} → {row['Destination']}"
    for row in route_legs.iter_rows(named=True)
]

incremental_cost_figure = go.Figure()

# Bar trace for incremental leg costs
incremental_cost_figure.add_trace(
    go.Bar(
        x=leg_labels,
        y=route_legs.get_column("Incremental_Cost").to_list(),
        name="Incremental Cost",
        marker=dict(
            color=route_legs.get_column("Incremental_Cost").to_list(),
            colorscale="Blues",
            line=dict(color="rgba(20, 40, 80, 0.85)", width=1),
        ),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Incremental Cost: $%{y:,.2f}<br>"
            "<extra></extra>"
        ),
    )
)

# Line trace for cumulative road trip cost
incremental_cost_figure.add_trace(
    go.Scatter(
        x=leg_labels,
        y=route_legs.get_column("Cumulative_Cost").to_list(),
        name="Cumulative Cost",
        mode="lines+markers",
        line=dict(color="#D97706", width=3),
        marker=dict(size=7, color="#D97706"),
        yaxis="y2",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Cumulative Cost: $%{y:,.2f}<br>"
            "<extra></extra>"
        ),
    )
)

# Update layout for chart readability and professional presentation
incremental_cost_figure.update_layout(
    title="A* Optimized Road Trip: Incremental and Cumulative Cost by Leg",
    xaxis_title="Road Trip Leg",
    yaxis=dict(title="Incremental Cost ($)", tickprefix="$", separatethousands=True),
    yaxis2=dict(
        title="Cumulative Cost ($)",
        tickprefix="$",
        separatethousands=True,
        overlaying="y",
        side="right",
    ),
    template="plotly_white",
    height=650,
    bargap=0.22,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=70, r=80, t=90, b=180),
)

incremental_cost_figure.update_xaxes(tickangle=45)
incremental_cost_figure.show()

### 5. Geospatial Visualization

The final stage of the notebook transforms the abstract mathematical output into an interactive, actionable map:

* **Geospatial Plotting**: We leverage **Folium** to plot every stadium on an OpenStreetMap interface. The "Home" base is clearly distinguished from the road destinations using dedicated markers and color-coding.
* **Sequential Path Rendering**: To visualize the optimized itinerary, we render `PolyLine` segments between nodes. These are enhanced with **directional arrows** to clearly indicate the tour flow.
* **Visual Analytics**: We implement a **color gradient** for the route segments—shifting from blue to orange—which provides a visual representation of the tour's progression over time.
* **Interactive Details**: Every element is interactive; clicking on any stadium marker or route segment triggers a custom HTML popup that displays the specific logistical costs, coordinate data, and leg-by-leg breakdowns, allowing for a deep dive into the optimization results.

In [ ]:
# Cell 5: Geospatial Map Visualization with Folium

from folium import plugins

def hex_gradient_color(start_hex: str, end_hex: str, fraction: float) -> str:
    """Interpolate a hexadecimal color between two endpoints for route visualization.

    Parameters
    ----------
    start_hex:
        Starting color in '#RRGGBB' format.
    end_hex:
        Ending color in '#RRGGBB' format.
    fraction:
        Value between 0.0 and 1.0 representing the gradient position.

    Returns
    -------
    str
        Interpolated color in '#RRGGBB' format.
    """
    bounded_fraction = min(1.0, max(0.0, fraction))
    start_rgb = tuple(int(start_hex[index : index + 2], 16) for index in (1, 3, 5))
    end_rgb = tuple(int(end_hex[index : index + 2], 16) for index in (1, 3, 5))
    mixed_rgb = tuple(
        int(round(start_channel + (end_channel - start_channel) * bounded_fraction))
        for start_channel, end_channel in zip(start_rgb, end_rgb)
    )
    return "#" + "".join(f"{channel:02x}" for channel in mixed_rgb)

# Map stadium names to coordinates for geospatial plotting
stadium_coordinates: Dict[str, Tuple[float, float]] = {
    row["City_Name"]: (float(row["Latitude"]), float(row["Longitude"]))
    for row in stadiums.iter_rows(named=True)
}

# Initialize Folium map centered on the centroid of all locations
centroid_latitude = float(stadiums.get_column("Latitude").mean())
centroid_longitude = float(stadiums.get_column("Longitude").mean())

road_trip_map = folium.Map(
    location=[centroid_latitude, centroid_longitude],
    zoom_start=4,
    tiles="OpenStreetMap",
    control_scale=True,
)

# Render stadium markers with detailed popups
for city in city_names:
    latitude, longitude = stadium_coordinates[city]
    is_home = city == HOME_CITY
    marker_color = "red" if is_home else "blue"
    marker_icon = "home" if is_home else "info-sign"
    fixed_cost = fixed_logistical_cost[city]

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; min-width: 210px;">
        <h4 style="margin-bottom: 6px;">{city}</h4>
        <div><b>Role:</b> {'Home Stadium' if is_home else 'Road Stadium'}</div>
        <div><b>Fixed logistical cost:</b> ${fixed_cost:,.2f}</div>
        <div><b>Latitude:</b> {latitude:.4f}</div>
        <div><b>Longitude:</b> {longitude:.4f}</div>
    </div>
    """

    folium.Marker(
        location=[latitude, longitude],
        tooltip=f"{city} | Fixed cost: ${fixed_cost:,.0f}",
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color=marker_color, icon=marker_icon),
    ).add_to(road_trip_map)

# Add numerical sequence markers to visualize trip order
for sequence_number, city in enumerate(optimal_route[:-1], start=1):
    latitude, longitude = stadium_coordinates[city]
    badge_background = "#B91C1C" if city == HOME_CITY else "#1D4ED8"
    folium.Marker(
        location=[latitude, longitude],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                background: {badge_background};
                color: white;
                border: 2px solid white;
                border-radius: 16px;
                box-shadow: 0 1px 4px rgba(0,0,0,0.35);
                font-size: 11px;
                font-weight: 700;
                height: 24px;
                line-height: 20px;
                text-align: center;
                width: 24px;">
                {sequence_number}
            </div>
            """
        ),
    ).add_to(road_trip_map)

# Draw route segments with a color gradient and directional arrows
number_of_legs = len(optimal_route) - 1
for leg_index, (origin_city, destination_city) in enumerate(zip(optimal_route[:-1], optimal_route[1:]), start=1):
    origin_coordinates = stadium_coordinates[origin_city]
    destination_coordinates = stadium_coordinates[destination_city]
    leg_cost = cost_lookup[(origin_city, destination_city)]
    leg_distance_cost = travel_cost_lookup[(origin_city, destination_city)]
    sequence_fraction = (leg_index - 1) / max(1, number_of_legs - 1)
    leg_color = hex_gradient_color("#2563EB", "#F97316", sequence_fraction)

    leg_popup_html = f"""
    <div style="font-family: Arial, sans-serif; min-width: 240px;">
        <h4 style="margin-bottom: 6px;">Leg {leg_index:02d}</h4>
        <div><b>Route:</b> {origin_city} → {destination_city}</div>
        <div><b>Total leg cost:</b> ${leg_cost:,.2f}</div>
        <div><b>Distance travel cost:</b> ${leg_distance_cost:,.2f}</div>
        <div><b>Destination fixed cost:</b> ${fixed_logistical_cost[destination_city]:,.2f}</div>
    </div>
    """

    route_segment = folium.PolyLine(
        locations=[origin_coordinates, destination_coordinates],
        color=leg_color,
        weight=5,
        opacity=0.88,
        tooltip=f"Leg {leg_index:02d}: {origin_city} → {destination_city} | ${leg_cost:,.0f}",
        popup=folium.Popup(leg_popup_html, max_width=320),
    ).add_to(road_trip_map)

    plugins.PolyLineTextPath(
        route_segment,
        "  ►  ",
        repeat=True,
        offset=8,
        attributes={
            "fill": leg_color,
            "font-weight": "bold",
            "font-size": "14",
        },
    ).add_to(road_trip_map)

# Auto-adjust map bounds to fit the full route
bounds = [[lat, lon] for lat, lon in stadium_coordinates.values()]
road_trip_map.fit_bounds(bounds, padding=(30, 30))
folium.LayerControl(collapsed=True).add_to(road_trip_map)

print("=" * 80)
print("Interactive Folium map rendered with optimized route sequence, gradient legs, and directional arrows.")
print("=" * 80)

road_trip_map